# 01 - Experiment Tracking (Lightweight)

---

## Learning Objectives

By the end of this notebook, you will be able to:

- Explain **why experiment tracking** is essential for reproducibility, comparison, and debugging
- Implement a lightweight **ExperimentLogger** class that writes to CSV
- Log multiple experiments with different hyperparameters
- Load, compare, and **visualize** experiment results from CSV files
- Optionally integrate with **TensorBoard** for richer visualization
- Apply best practices for experiment organization

## Prerequisites

- Basic Python (classes, file I/O, CSV)
- PyTorch fundamentals (training loops, loss functions)
- Familiarity with `matplotlib` for plotting

## Table of Contents

1. [Why Track Experiments?](#1-why-track-experiments)
2. [Simple CSV Logger: ExperimentLogger](#2-simple-csv-logger-experimentlogger)
3. [Logging Multiple Experiments](#3-logging-multiple-experiments)
4. [Loading and Comparing Experiments](#4-loading-and-comparing-experiments)
5. [Plotting Comparison Charts](#5-plotting-comparison-charts)
6. [Optional: TensorBoard Integration](#6-optional-tensorboard-integration)
7. [Best Practices for Experiment Organization](#7-best-practices-for-experiment-organization)
8. [Exercise: Hyperparameter Sweep](#8-exercise-hyperparameter-sweep)
9. [Common Mistakes & Debugging Tips](#9-common-mistakes--debugging-tips)

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

import os
import csv
import json
import time
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

set_global_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

---
## 1. Why Track Experiments?

Deep learning projects involve dozens (or hundreds) of training runs with different:
- **Hyperparameters**: learning rate, batch size, hidden dimensions, dropout, etc.
- **Architectures**: number of layers, activation functions, normalization
- **Data configurations**: train/val splits, augmentation strategies

Without tracking, you quickly lose track of what worked and what did not.

### Three pillars of experiment tracking:

| Pillar | Why it matters |
|---|---|
| **Reproducibility** | Re-run any past experiment with identical results |
| **Comparison** | Compare metrics across runs to find the best configuration |
| **Debugging** | Diagnose training failures by examining loss curves and hyperparams |

### What to log:
- **Per-epoch metrics**: train loss, validation loss, accuracy, etc.
- **Hyperparameters**: learning rate, batch size, model architecture choices
- **Metadata**: timestamp, random seed, hardware info, training duration
- **Artifacts**: model checkpoints, sample predictions

---
## 2. Simple CSV Logger: ExperimentLogger

We build a lightweight logger that:
1. Creates a directory for each experiment
2. Saves hyperparameters as JSON
3. Logs per-epoch metrics to a CSV file
4. Records a summary upon completion

In [ ]:
class ExperimentLogger:
    """Lightweight experiment tracker that writes to CSV and JSON files.
    
    Directory structure:
        base_dir/
            experiment_name/
                hyperparams.json
                metrics.csv
                summary.json
    """
    
    def __init__(self, experiment_name: str, base_dir: str = "experiments",
                 hyperparams: Optional[Dict[str, Any]] = None):
        self.experiment_name = experiment_name
        self.exp_dir = Path(base_dir) / experiment_name
        self.exp_dir.mkdir(parents=True, exist_ok=True)
        
        self.metrics_path = self.exp_dir / "metrics.csv"
        self.hyperparams_path = self.exp_dir / "hyperparams.json"
        self.summary_path = self.exp_dir / "summary.json"
        
        self.start_time = datetime.now()
        self.metrics_history: List[Dict[str, Any]] = []
        self._csv_writer = None
        self._csv_file = None
        self._fieldnames = None
        
        # Save hyperparameters
        self.hyperparams = hyperparams or {}
        self.hyperparams["experiment_name"] = experiment_name
        self.hyperparams["start_time"] = self.start_time.isoformat()
        with open(self.hyperparams_path, "w") as f:
            json.dump(self.hyperparams, f, indent=2, default=str)
        
        print(f"Experiment '{experiment_name}' initialized at {self.exp_dir}")
    
    def log_epoch(self, epoch: int, **metrics) -> None:
        """Log metrics for a single epoch.
        
        Args:
            epoch: Epoch number.
            **metrics: Keyword arguments for metric values (e.g., train_loss=0.5).
        """
        row = {"epoch": epoch, **metrics}
        self.metrics_history.append(row)
        
        # Initialize CSV on first call
        if self._csv_writer is None:
            self._fieldnames = list(row.keys())
            self._csv_file = open(self.metrics_path, "w", newline="")
            self._csv_writer = csv.DictWriter(self._csv_file,
                                               fieldnames=self._fieldnames)
            self._csv_writer.writeheader()
        
        self._csv_writer.writerow(row)
        self._csv_file.flush()  # Ensure data is written immediately
    
    def finish(self, **extra_summary) -> Dict[str, Any]:
        """Finalize the experiment and write a summary.
        
        Args:
            **extra_summary: Additional key-value pairs to include in the summary.
            
        Returns:
            Summary dictionary.
        """
        if self._csv_file is not None:
            self._csv_file.close()
        
        duration = (datetime.now() - self.start_time).total_seconds()
        
        # Compute summary statistics
        summary = {
            "experiment_name": self.experiment_name,
            "total_epochs": len(self.metrics_history),
            "duration_seconds": round(duration, 2),
            "start_time": self.start_time.isoformat(),
            "end_time": datetime.now().isoformat(),
        }
        
        # Best metrics
        if self.metrics_history:
            val_losses = [m.get("val_loss") for m in self.metrics_history
                          if m.get("val_loss") is not None]
            if val_losses:
                best_idx = int(np.argmin(val_losses))
                summary["best_val_loss"] = val_losses[best_idx]
                summary["best_epoch"] = self.metrics_history[best_idx]["epoch"]
            
            # Final train loss
            last = self.metrics_history[-1]
            if "train_loss" in last:
                summary["final_train_loss"] = last["train_loss"]
        
        summary.update(extra_summary)
        summary["hyperparams"] = self.hyperparams
        
        with open(self.summary_path, "w") as f:
            json.dump(summary, f, indent=2, default=str)
        
        print(f"Experiment '{self.experiment_name}' finished. "
              f"Duration: {duration:.1f}s, Epochs: {summary['total_epochs']}")
        return summary
    
    @staticmethod
    def load_metrics(experiment_dir: str) -> List[Dict[str, float]]:
        """Load metrics from a CSV file.
        
        Args:
            experiment_dir: Path to the experiment directory.
            
        Returns:
            List of dicts, one per epoch.
        """
        metrics_path = Path(experiment_dir) / "metrics.csv"
        rows = []
        with open(metrics_path, "r") as f:
            reader = csv.DictReader(f)
            for row in reader:
                # Convert numeric strings to floats
                parsed = {}
                for k, v in row.items():
                    try:
                        parsed[k] = float(v)
                    except (ValueError, TypeError):
                        parsed[k] = v
                rows.append(parsed)
        return rows
    
    @staticmethod
    def load_hyperparams(experiment_dir: str) -> Dict[str, Any]:
        """Load hyperparameters from a JSON file."""
        hp_path = Path(experiment_dir) / "hyperparams.json"
        with open(hp_path, "r") as f:
            return json.load(f)
    
    @staticmethod
    def load_summary(experiment_dir: str) -> Dict[str, Any]:
        """Load summary from a JSON file."""
        summary_path = Path(experiment_dir) / "summary.json"
        with open(summary_path, "r") as f:
            return json.load(f)

In [ ]:
# Quick test of the ExperimentLogger
logger = ExperimentLogger(
    experiment_name="test_run_001",
    base_dir="experiments",
    hyperparams={"lr": 0.01, "batch_size": 32, "hidden_dim": 64}
)

# Simulate 5 epochs of training
for epoch in range(1, 6):
    train_loss = 1.0 / epoch + np.random.normal(0, 0.05)
    val_loss = 1.0 / epoch + np.random.normal(0, 0.08)
    accuracy = 0.5 + 0.1 * epoch + np.random.normal(0, 0.02)
    logger.log_epoch(epoch, train_loss=round(train_loss, 4),
                     val_loss=round(val_loss, 4),
                     accuracy=round(accuracy, 4))

summary = logger.finish()
print("\nSummary:")
for k, v in summary.items():
    if k != "hyperparams":
        print(f"  {k}: {v}")

---
## 3. Logging Multiple Experiments

Now let us run multiple experiments with different hyperparameters and log each one. We will use a simple MLP on synthetic data.

In [ ]:
# Create synthetic dataset
set_global_seed(42)

N_SAMPLES = 1000
N_FEATURES = 10
N_CLASSES = 3

X = torch.randn(N_SAMPLES, N_FEATURES)
# Generate labels based on a simple rule
true_w = torch.randn(N_FEATURES, N_CLASSES)
logits = X @ true_w
y = logits.argmax(dim=1)

# Train / validation split
split = int(0.8 * N_SAMPLES)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

print(f"Training samples: {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Features: {N_FEATURES}, Classes: {N_CLASSES}")

In [ ]:
class SimpleMLP(nn.Module):
    """Simple multi-layer perceptron for classification."""
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int,
                 dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )
    
    def forward(self, x):
        return self.net(x)


def train_experiment(hyperparams: Dict[str, Any], X_train, y_train,
                     X_val, y_val, exp_name: str,
                     base_dir: str = "experiments") -> Dict:
    """Train a model and log all metrics.
    
    Args:
        hyperparams: Dict with keys: lr, batch_size, hidden_dim, dropout, epochs.
        X_train, y_train: Training data tensors.
        X_val, y_val: Validation data tensors.
        exp_name: Experiment name for logging.
        base_dir: Base directory for experiment logs.
    
    Returns:
        Summary dictionary.
    """
    set_global_seed(42)
    
    logger = ExperimentLogger(exp_name, base_dir=base_dir,
                              hyperparams=hyperparams)
    
    # Create data loaders
    train_ds = TensorDataset(X_train, y_train)
    val_ds = TensorDataset(X_val, y_val)
    train_loader = DataLoader(train_ds, batch_size=hyperparams["batch_size"],
                              shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=hyperparams["batch_size"])
    
    # Create model
    model = SimpleMLP(
        input_dim=X_train.shape[1],
        hidden_dim=hyperparams["hidden_dim"],
        output_dim=N_CLASSES,
        dropout=hyperparams.get("dropout", 0.0),
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=hyperparams["lr"])
    loss_fn = nn.CrossEntropyLoss()
    
    for epoch in range(1, hyperparams["epochs"] + 1):
        # Train
        model.train()
        total_train_loss = 0.0
        n_train = 0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            out = model(X_batch)
            loss = loss_fn(out, y_batch)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
            n_train += 1
        train_loss = total_train_loss / n_train
        
        # Validate
        model.eval()
        total_val_loss = 0.0
        correct = 0
        total = 0
        n_val = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                out = model(X_batch)
                loss = loss_fn(out, y_batch)
                total_val_loss += loss.item()
                n_val += 1
                preds = out.argmax(dim=1)
                correct += (preds == y_batch).sum().item()
                total += y_batch.size(0)
        val_loss = total_val_loss / n_val
        val_acc = correct / total
        
        logger.log_epoch(epoch, train_loss=round(train_loss, 4),
                         val_loss=round(val_loss, 4),
                         val_accuracy=round(val_acc, 4))
    
    return logger.finish()

In [ ]:
# Define experiment configurations
experiments = {
    "exp_lr001_h64": {"lr": 0.01, "batch_size": 32, "hidden_dim": 64,
                       "dropout": 0.0, "epochs": 20},
    "exp_lr001_h128": {"lr": 0.01, "batch_size": 32, "hidden_dim": 128,
                        "dropout": 0.0, "epochs": 20},
    "exp_lr0001_h64": {"lr": 0.001, "batch_size": 32, "hidden_dim": 64,
                        "dropout": 0.0, "epochs": 20},
    "exp_lr001_h64_drop": {"lr": 0.01, "batch_size": 32, "hidden_dim": 64,
                            "dropout": 0.2, "epochs": 20},
}

# Run all experiments
summaries = {}
for name, hparams in experiments.items():
    print(f"\n{'='*60}")
    summaries[name] = train_experiment(hparams, X_train, y_train,
                                       X_val, y_val, name)

print(f"\n{'='*60}")
print("All experiments completed!")

---
## 4. Loading and Comparing Experiments

Now let us load the logged data and compare experiments.

In [ ]:
# Load and compare all experiments
print(f"{'Experiment':<25} {'Best Val Loss':>14} {'Best Epoch':>11} {'Final Train Loss':>17}")
print("-" * 70)

for name in experiments:
    exp_dir = Path("experiments") / name
    summary = ExperimentLogger.load_summary(str(exp_dir))
    best_val = summary.get("best_val_loss", "N/A")
    best_ep = summary.get("best_epoch", "N/A")
    final_train = summary.get("final_train_loss", "N/A")
    print(f"{name:<25} {best_val:>14.4f} {best_ep:>11.0f} {final_train:>17.4f}")

In [ ]:
# Load metrics for detailed comparison
all_metrics = {}
for name in experiments:
    exp_dir = Path("experiments") / name
    all_metrics[name] = ExperimentLogger.load_metrics(str(exp_dir))

# Print first 3 epochs of first experiment as a sample
sample_name = list(experiments.keys())[0]
print(f"Sample metrics from '{sample_name}':")
for row in all_metrics[sample_name][:3]:
    print(f"  {row}")

---
## 5. Plotting Comparison Charts

Visualizing experiments side-by-side is the fastest way to compare.

In [ ]:
def plot_experiment_comparison(all_metrics: Dict[str, List[Dict]],
                                metric_name: str = "val_loss",
                                title: str = "Experiment Comparison",
                                figsize=(10, 5)):
    """Plot a metric across multiple experiments.
    
    Args:
        all_metrics: Dict mapping experiment name -> list of epoch dicts.
        metric_name: Which metric to plot.
        title: Plot title.
        figsize: Figure size.
    """
    fig, ax = plt.subplots(figsize=figsize)
    
    for name, metrics in all_metrics.items():
        epochs = [m["epoch"] for m in metrics]
        values = [m[metric_name] for m in metrics]
        ax.plot(epochs, values, marker="o", markersize=3, label=name)
    
    ax.set_xlabel("Epoch")
    ax.set_ylabel(metric_name.replace("_", " ").title())
    ax.set_title(title)
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# Plot validation loss comparison
plot_experiment_comparison(all_metrics, metric_name="val_loss",
                           title="Validation Loss Comparison")

In [ ]:
# Plot training loss comparison
plot_experiment_comparison(all_metrics, metric_name="train_loss",
                           title="Training Loss Comparison")

In [ ]:
# Plot validation accuracy comparison
plot_experiment_comparison(all_metrics, metric_name="val_accuracy",
                           title="Validation Accuracy Comparison")

In [ ]:
# Bar chart: best validation loss per experiment
fig, ax = plt.subplots(figsize=(8, 4))

names = []
best_vals = []
for name in experiments:
    exp_dir = Path("experiments") / name
    summary = ExperimentLogger.load_summary(str(exp_dir))
    names.append(name)
    best_vals.append(summary["best_val_loss"])

colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(names)))
bars = ax.bar(names, best_vals, color=colors, edgecolor="black")

# Highlight the best
best_idx = int(np.argmin(best_vals))
bars[best_idx].set_edgecolor("red")
bars[best_idx].set_linewidth(2)

ax.set_ylabel("Best Validation Loss")
ax.set_title("Best Validation Loss per Experiment")
ax.tick_params(axis="x", rotation=25)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print(f"\nBest experiment: {names[best_idx]} (val_loss={best_vals[best_idx]:.4f})")

---
## 6. Optional: TensorBoard Integration

TensorBoard provides interactive visualizations. This section is **optional** -- it only runs if `tensorboard` is installed.

In [ ]:
try:
    from torch.utils.tensorboard import SummaryWriter
    TB_AVAILABLE = True
except ImportError:
    TB_AVAILABLE = False

print(f"TensorBoard available: {TB_AVAILABLE}")

In [ ]:
class TensorBoardLogger:
    """Wrapper that logs to both CSV and TensorBoard (if available)."""
    
    def __init__(self, experiment_name: str, base_dir: str = "experiments",
                 hyperparams: Optional[Dict[str, Any]] = None):
        # Always create the CSV logger
        self.csv_logger = ExperimentLogger(experiment_name, base_dir, hyperparams)
        
        # Optionally create TensorBoard writer
        self.tb_writer = None
        if TB_AVAILABLE:
            tb_dir = Path(base_dir) / experiment_name / "tensorboard"
            self.tb_writer = SummaryWriter(log_dir=str(tb_dir))
            if hyperparams:
                self.tb_writer.add_text("hyperparams",
                                         json.dumps(hyperparams, indent=2))
            print(f"  TensorBoard logs: {tb_dir}")
    
    def log_epoch(self, epoch: int, **metrics) -> None:
        """Log to both CSV and TensorBoard."""
        self.csv_logger.log_epoch(epoch, **metrics)
        if self.tb_writer is not None:
            for key, value in metrics.items():
                self.tb_writer.add_scalar(key, value, epoch)
    
    def finish(self, **extra_summary) -> Dict[str, Any]:
        """Finalize both loggers."""
        if self.tb_writer is not None:
            self.tb_writer.close()
        return self.csv_logger.finish(**extra_summary)


# Demo: run one experiment with TensorBoard logging
if TB_AVAILABLE:
    tb_logger = TensorBoardLogger(
        "tb_demo", base_dir="experiments",
        hyperparams={"lr": 0.01, "hidden_dim": 64}
    )
    for epoch in range(1, 11):
        tb_logger.log_epoch(epoch,
                            train_loss=round(1.0 / epoch, 4),
                            val_loss=round(1.0 / epoch + 0.05, 4))
    tb_logger.finish()
    print("\nTo view TensorBoard, run: tensorboard --logdir experiments/tb_demo/tensorboard")
else:
    print("TensorBoard not installed. Install with: pip install tensorboard")
    print("The CSV logger works independently and does not require TensorBoard.")

---
## 7. Best Practices for Experiment Organization

### Directory structure
```
experiments/
    experiment_001_baseline/
        hyperparams.json
        metrics.csv
        summary.json
        checkpoints/
            best_model.pt
        tensorboard/   (optional)
    experiment_002_higher_lr/
        ...
```

### Naming conventions
- Use descriptive names: `exp_lr001_h128_drop02` rather than `exp_42`
- Include timestamps if running many: `2025_01_15_14h30_lr001`
- Use a consistent scheme across all experiments

### What to always log
- **Hyperparameters**: every tuneable value (learning rate, architecture, augmentation)
- **Random seeds**: for full reproducibility
- **Environment**: Python version, PyTorch version, GPU model
- **Per-epoch metrics**: at minimum train loss and val loss
- **Git commit hash**: tie the experiment to a specific code version

### Tips
- Add your `experiments/` directory to `.gitignore` (logs can be large)
- Keep a separate `experiments_index.csv` that summarizes all runs
- Use version control for code, experiment tracking for results
- Never modify logged files after the fact -- treat them as immutable

---
## 8. Exercise: Hyperparameter Sweep

**Task:** Perform a grid search over learning rates and hidden dimensions. Use the `ExperimentLogger` to track all runs, then find the best configuration.

1. Define a grid: `lr` in `[0.1, 0.01, 0.001]`, `hidden_dim` in `[32, 64, 128]`
2. Train each configuration for 15 epochs
3. Log all metrics using `ExperimentLogger`
4. Load results and create a summary table
5. Identify the best configuration by validation loss

```python
# Your code here
learning_rates = [0.1, 0.01, 0.001]
hidden_dims = [32, 64, 128]

# TODO: Loop over all combinations
# TODO: Train and log each experiment
# TODO: Load results and print comparison table
# TODO: Find and print the best configuration
```

In [ ]:
# Try the exercise yourself before looking at the solution below!








### Solution

In [ ]:
# ----- Solution -----

learning_rates = [0.1, 0.01, 0.001]
hidden_dims = [32, 64, 128]

sweep_summaries = {}

for lr in learning_rates:
    for hd in hidden_dims:
        exp_name = f"sweep_lr{lr}_h{hd}"
        hparams = {
            "lr": lr,
            "batch_size": 32,
            "hidden_dim": hd,
            "dropout": 0.0,
            "epochs": 15,
        }
        print(f"Running {exp_name}...")
        summary = train_experiment(
            hparams, X_train, y_train, X_val, y_val,
            exp_name, base_dir="experiments"
        )
        sweep_summaries[exp_name] = summary

print(f"\n{'='*70}")
print(f"{'Experiment':<25} {'LR':>8} {'Hidden':>8} {'Best Val Loss':>14} {'Best Epoch':>11}")
print("-" * 70)

best_name = None
best_loss = float("inf")

for name, summary in sweep_summaries.items():
    hp = summary["hyperparams"]
    val = summary.get("best_val_loss", float("inf"))
    ep = summary.get("best_epoch", -1)
    print(f"{name:<25} {hp['lr']:>8.4f} {hp['hidden_dim']:>8d} {val:>14.4f} {ep:>11.0f}")
    if val < best_loss:
        best_loss = val
        best_name = name

print(f"\nBest configuration: {best_name} (val_loss={best_loss:.4f})")
print(f"Hyperparams: {sweep_summaries[best_name]['hyperparams']}")

In [ ]:
# Visualize sweep results as a heatmap
fig, ax = plt.subplots(figsize=(7, 5))

results_matrix = np.zeros((len(learning_rates), len(hidden_dims)))
for i, lr in enumerate(learning_rates):
    for j, hd in enumerate(hidden_dims):
        exp_name = f"sweep_lr{lr}_h{hd}"
        results_matrix[i, j] = sweep_summaries[exp_name].get("best_val_loss", 0)

im = ax.imshow(results_matrix, cmap="RdYlGn_r", aspect="auto")
ax.set_xticks(range(len(hidden_dims)))
ax.set_xticklabels(hidden_dims)
ax.set_yticks(range(len(learning_rates)))
ax.set_yticklabels(learning_rates)
ax.set_xlabel("Hidden Dimension")
ax.set_ylabel("Learning Rate")
ax.set_title("Hyperparameter Sweep: Best Validation Loss")

# Annotate cells
for i in range(len(learning_rates)):
    for j in range(len(hidden_dims)):
        ax.text(j, i, f"{results_matrix[i, j]:.3f}",
                ha="center", va="center", fontsize=11, fontweight="bold")

fig.colorbar(im, ax=ax, label="Best Val Loss")
plt.tight_layout()
plt.show()

In [ ]:
# Clean up experiment directories
import shutil
if os.path.exists("experiments"):
    shutil.rmtree("experiments")
    print("Cleaned up experiments directory.")

---
## 9. Common Mistakes & Debugging Tips

**1. Not flushing log files**
- If training crashes, you lose unflushed data. Always call `file.flush()` after each write.
- Our `ExperimentLogger` flushes after every `log_epoch()` call.

**2. Overwriting experiment logs**
- Use unique experiment names (timestamps help).
- Check if the directory exists before creating a new experiment.

**3. Logging too infrequently**
- Log every epoch at minimum. For long epochs, consider logging every N batches.
- More granular logging helps debug mid-epoch crashes.

**4. Not logging hyperparameters**
- You cannot reproduce a run if you do not know the hyperparameters.
- Log everything: learning rate, batch size, architecture, seed, optimizer settings.

**5. Confusing CSV column types**
- CSV stores everything as strings. Remember to convert when loading.
- Our `load_metrics()` method handles this automatically.

**6. Ignoring failed experiments**
- Failed runs carry valuable information (e.g., learning rate too high causes NaN).
- Log the failure reason in the summary before discarding.

**7. Mixing up train and validation metrics**
- Use clear, consistent naming: `train_loss`, `val_loss`, `train_acc`, `val_acc`.
- Never compare train metrics of one run with validation metrics of another.

---

**Next notebook:** [02 - Model Saving, Loading & Inference Pipeline](02_Model_Saving_Loading_Inference_Pipeline.ipynb) -- we cover checkpointing, loading models, and building complete inference pipelines.